In [ ]:
import os.path

import scipy.stats as stats
import multiprocessing as mp

from analysis import KMCv2Analyzer
from kmc import KMCv2Runner, KMCv2Viewer, get_params_from_csv
from pathlib import Path

import numpy as np
import sqlite3 as sql

In [ ]:
out_dir = Path('../temp/output')
runs_iterable = range(10)
just_analyze = True
steps = 100_000_000
sigma_iterable = [x * 0.02 for x in range(8)]
dimensions = [
    [1000],
    [30,30],
    [10, 10, 10],
    [6, 6, 6, 6],
    [5, 5, 5, 5, 5]
]
lattice_vectors = [
    [1],
    [1,1],
    [1,1,1],
	[1,1,1,1],
	[1,1,1,1,1]
]

In [ ]:
init_params, saddle_params = get_params_from_csv('../csv/1.csv', 'utf-8')
init_mean = init_params[0]
saddle_mean = saddle_params[0]
sets = {}

init_params = [[init_mean, sigma] for sigma in sigma_iterable for _ in dimensions]
saddle_params = [[saddle_mean, sigma] for sigma in sigma_iterable for _ in dimensions]
dims_params = [dims for _ in sigma_iterable for dims in dimensions]
lattice_params = [lat for _ in sigma_iterable for lat in lattice_vectors]

In [ ]:
# Running
if not just_analyze:
    runner = KMCv2Runner(out_dir, dims_params, lattice_params, init_params, saddle_params, 1000)
    runner.run(steps, runs_iterable)
viewer = KMCv2Viewer(out_dir, dims_params, lattice_params, init_params, saddle_params, 1000)
slopes = viewer.get_slopes()


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
means = {}
stds = {}
divisors = {}
for d in [tuple(x) for x in dimensions]:
    if slopes[d][(0.0,0.0)] is None or slopes[d][(0.0,0.0)].size <= 0:
        divisors[d] = 1
        continue
    divisors[d] = np.mean(slopes[d][(0.0,0.0)])
for d in [tuple(x) for x in dimensions[1:]]:
    means[d] = []
    stds[d] = []
    for i in [(x,x) for x in sigma_iterable]:
        means[d].append(np.mean(slopes[d][i])/ divisors[d])
        stds[d].append(np.std(slopes[d][i]) / divisors[d])
    plt.errorbar(sigma_iterable, means[d], yerr=stds[d], label=f'{len(d)}D')

plt.xlabel('Sigma')
plt.ylabel('Diffusivity')
plt.legend()
plt.show()

In [ ]:
arr_str = KMCv2Runner.arr_str
def get_con(i, which, run):
    db_path = (
        f'{out_dir}/kmc/'
        f'{arr_str(dimensions[which])}D'
        f'{arr_str(lattice_vectors[which])}V'
        f'{arr_str((init_mean, i))}I'
        f'{arr_str((saddle_mean, i))}S'
        f'-{run}.db'
    )
    if not os.path.exists(db_path):
        print(f'{db_path} does not exist')
        return None
    return sql.connect(db_path)

In [ ]:
def make_plot(run):
    which = 0
    i = 0.0
    ret = get_con(i, which, run)
    if ret is None:
        return
    con = ret
    max_t = float(con.execute('SELECT MAX(time) FROM kmc').fetchone()[0])
    a = KMCv2Analyzer(con,con)
    dt, pos = a._evenly_spaced_pos(100_000,max_t, len(dimensions[which]))
    plt.plot(dt, pos)
    plt.title(arr_str(dimensions[which]))
    plt.savefig(f'../plots/{arr_str(dimensions[which])}-{i}-{run}.png')
    plt.show()
    return i, dt, pos

with mp.Pool() as pool:
    pass
    #res = pool.map(make_plot, runs_iterable)

In [ ]:
def make_grid_plots(share: bool = True):
    for i in [sigma_iterable[x] for x in range(0, len(sigma_iterable), 1)]:
        print(f'----SIGMA: {i:.2f}---')
        fig, axs = plt.subplots(len(runs_iterable), len(dimensions),sharey=share, sharex=share)
        fig.set_figheight(10)
        fig.set_figwidth(23)
        for run in runs_iterable:
        #for run in [0,1]:
            for which in range(len(dimensions)):
                con = get_con(i, which, run)
                if con is None:
                    continue
                res = np.array(con.execute('SELECT dt, msd FROM msd ORDER BY dt').fetchall())
                dt, msd = res.transpose()
                linreg = stats.linregress(dt, msd)
                axs[run, which].plot(dt, msd)
                fitted = dt * linreg.slope + linreg.intercept
                axs[run, which].plot(dt, fitted, color='red')
                axs[run, which].set_title(f'{which+1}D ,R={linreg.rvalue:.3f}')
        #plt.savefig(f'../plots/{arr_str(dimensions[which])}-{i}-msd.png')
        plt.show()

In [ ]:
def msd(con:sql.Connection, count, which):
    max_t = float(con.execute('SELECT MAX(time) FROM kmc').fetchone()[0])
    a = KMCv2Analyzer(con,con)

    dt, positions = a._evenly_spaced_pos(count,max_t, len(dimensions[which]))
    dt = [float (x) for x in dt]
    pos = dict(zip(dt,positions))

    max_dt = max(dt)

    msd = []
    for dt_us in dt:
        disp_square = []
        for dt_start, pos_us in pos.items():
            dt_end = dt_start + dt_us
            if dt_end > max_dt or dt_end not in pos:
                continue
            disp = pos_us - pos[dt_end]
            disp_square.append(disp**2)
        msd.append(np.mean(disp_square))

    return msd, dt

    a.calculate_msds(count, len(dimensions[which]))
    return a.get_msds()

msd_vals = msd(get_con(0.0,0,0),10_000, 0)


In [ ]:
plt.plot(msd_vals[1], msd_vals[0])

In [ ]:
for i in [sigma_iterable[x] for x in range(0, len(sigma_iterable), 1)]:
    print(f'----SIGMA: {i:.2f}---')
    fig, axs = plt.subplots(len(runs_iterable), len(dimensions))
    #fig, axs = plt.subplots(2,len(dimensions))
    fig.set_figheight(30)
    fig.set_figwidth(50)
    for which in range(len(dimensions)):
        axs[0, which].set_title(f'{which+1}D')
    #for run in [0,1]:
    for run in runs_iterable:
        for which in range(len(dimensions)):
            con = get_con(i, which, run)
            if con is None:
                continue
            res = np.array(con.execute('SELECT dt, msd FROM msd ORDER BY dt').fetchall())
            dt, msd = res.transpose()
            linreg = stats.linregress(dt, msd)
            axs[run, which].plot(dt, msd)
            fitted = dt * linreg.slope + linreg.intercept
            axs[run, which].plot(dt, fitted, color='red')
            #axs[run, which].set_title(f'{which+1}D ,R={linreg.rvalue:.3f}')
    #plt.savefig(f'../plots/{arr_str(dimensions[which])}-{i}-msd.png')
    plt.tight_layout()
    plt.show()